In [1]:
%load_ext autoreload
%autoreload 2

In [83]:
from rdkit import Chem
from rdkit.Chem import rdChemReactions
from rdkit.Chem import Draw
from rdkit.Chem import AllChem, rdchem

import re
import importlib
import numpy as np
from IPython.display import display, Markdown
import ipywidgets as widgets
import importlib.util, sys, types
import ipywidgets as widgets
from IPython.display import display

In [84]:
# Extract atomic symbol
def extract_symbol(s):
    if s.startswith("#"):
        try:
            num = int(re.findall(r"#(\d+)", s)[0])
            return Chem.GetPeriodicTable().GetElementSymbol(num)
        except:
            return "*"
    match = re.match(r"([A-Z][a-z]?)", s)
    return match.group(1) if match else "*"

# Label atoms
def set_query_atom_labels(mol, mode='clean'):
    for atom in mol.GetAtoms():
        if not atom.HasQuery():
            continue

        smarts = Chem.MolFragmentToSmarts(mol, atomsToUse=[atom.GetIdx()])
        if not (smarts.startswith("[") and smarts.endswith("]")):
            atom.SetProp("_displayLabel", atom.GetSymbol())
            continue

        inner = smarts[1:-1]

        if mode == 'query':
            atom.SetProp("_displayLabel", inner)
            continue

        # clean mode
        parts = re.split(r"[;&]", inner)
        symbol_raw = parts[0]
        if ":" in symbol_raw:
            symbol_raw = symbol_raw.split(":")[0]
        symbol = extract_symbol(symbol_raw)

        hcount = None
        charge = None
        for part in parts[1:]:
            if part.startswith("H"):
                try:
                    hcount = int(part[1:])
                except:
                    pass
            elif part.startswith("+") or part.startswith("-"):
                match = re.match(r"[+-]\d*", part)
                if match:
                    charge = match.group()

        label = symbol
        if hcount == 1:
            label += "H"
        elif hcount and hcount > 1:
            label += f"H{hcount}"
        if charge not in (None, "+0", "-0", "0"):
            label += charge

        atom.SetProp("_displayLabel", label)

# Draw reaction
def draw_reaction_smarts(smarts, mode='clean'):
    try:
        rxn = rdChemReactions.ReactionFromSmarts(smarts, useSmiles=False)
        for mol in rxn.GetReactants():
            set_query_atom_labels(mol, mode)
        for mol in rxn.GetProducts():
            set_query_atom_labels(mol, mode)
        return Draw.ReactionToImage(rxn, subImgSize=(300, 200))
    except Exception as e:
        return f"Error: {e}"

In [80]:
import groups
importlib.reload(groups)
import elem_steps
steps = elem_steps.steps
importlib.reload(elem_steps)
import reaction_templates
importlib.reload(reaction_templates)

<module 'reaction_templates' from '/home/erick/ColeyLinux/Mechanistic_dataset/new_templates/reaction_templates.py'>

In [81]:
class_reaction_templates = reaction_templates.class_reaction_templates

# Count how many reaction classes are available:
all_classes = []
for class_key in class_reaction_templates:
    # class_key might be a tuple of class names or a single string
    if isinstance(class_key, tuple):
        all_classes.extend(list(class_key))
    else:
        all_classes.append(class_key)
all_classes_count = len(all_classes)
print(f"Loaded reaction templates for {all_classes_count} reaction classes.")

Loaded reaction templates for 68 reaction classes.


In [82]:
reaction_dict = class_reaction_templates 

class_dropdown = widgets.Dropdown(options=list(reaction_dict.keys()), description='Class:')
mech_dropdown = widgets.Dropdown(description='Mechanism:')
step_dropdown = widgets.Dropdown(description='Step:')
mode_dropdown = widgets.Dropdown(options=['clean', 'query'], value='clean', description='Mode:')
output = widgets.Output()

# Update mechanism list
def update_mechanisms(change):
    mech_dict = reaction_dict[change['new']]
    mech_dropdown.options = list(mech_dict.keys())
    if mech_dropdown.options:
        mech_dropdown.value = mech_dropdown.options[0]
        update_steps({'new': mech_dropdown.value})


# Update step list
def update_steps(change):
    mech_data = reaction_dict[class_dropdown.value][change['new']]
#     print("Selected class:", class_dropdown.value)
#     print("Selected reaction:", change['new'])
#     print("Stages available:", list(mech_data.get("Stages", {}).keys()))
    step_dropdown.options = list(mech_data.get("Stages", {}).keys())
    if step_dropdown.options:
        step_dropdown.value = step_dropdown.options[0]

# Display selected step
def display_step(change=None):
    with output:
        output.clear_output()
        cls = class_dropdown.value
        mech = mech_dropdown.value
        step = step_dropdown.value
        mode = mode_dropdown.value
        data = reaction_dict[cls][mech]
        reagent = data.get("Reagent", "N/A")
        exclude = data.get("Exclude_reagent", "N/A")
        stage_data = data["Stages"].get(step, {})

        if isinstance(stage_data, dict) and "Templates" in stage_data:
            templates = stage_data["Templates"]
            description = stage_data.get("Description", "")
        else:
            templates = [stage_data] if isinstance(stage_data, str) else []
            description = ""

        print(f"[{cls} - {mech}]")
        print(f"Reagent: {reagent}")
        print(f"Exclude: {exclude}")
        print(f"Step: {step}")
        if description:
            print(f"Description: {description}")
        for i, tpl in enumerate(templates):
            print(f"Template {i+1}: {tpl}")
            img = draw_reaction_smarts(tpl, mode)
            if isinstance(img, str):
                print(img)
            else:
                display(img)

# Bind events
class_dropdown.observe(update_mechanisms, names='value')
mech_dropdown.observe(update_steps, names='value')
step_dropdown.observe(display_step, names='value')
mode_dropdown.observe(display_step, names='value')

# Trigger initial updates
update_mechanisms({'new': class_dropdown.value})
update_steps({'new': mech_dropdown.value})
display_step()

# Show UI
ui = widgets.VBox([
    class_dropdown,
    mech_dropdown,
    step_dropdown,
    mode_dropdown,
    output
])
display(ui)
